<a href="https://colab.research.google.com/github/SenaKahveci/HW1/blob/master/1_Housing_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What kinds of homes are in Saint John, and what are the financial circumstances of the households living in them?

This code expects the English, comprehensive 2021 Census Profile CSV, with columns such as DGUID, CHARACTERISTIC_NAME, and C1_COUNT_TOTAL. Statistics Canada offers several download formats; the smaller “data as displayed” download may have a different structure. Use the official Census Profile downloads and select the New Brunswick comprehensive file containing provinces and census subdivisions.

| What we want to know | What we will measure |
|---|---|
| How many homes are there? | Number of occupied dwellings |
| What types of homes are there? | Percentage of single-detached houses and apartments |
| How old are the homes? | Percentage built before 1981 |
| What condition are they in? | Percentage requiring major repairs |
| Do households own or rent? | Percentage of owners and renters |
| What financial resources do households have? | Median after-tax household income |
| How much pressure do housing costs create? | Percentage spending at least 30% of income on shelter |

We will calculate these figures for Saint John city and put New Brunswick beside them for comparison. This will help us understand whether Saint John has, for example, older housing or greater housing affordability pressure than the province overall.

After we understand the city overall, we can repeat the same exercise for smaller areas within Saint John to see where older housing and financial constraints are concentrated.


In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

Step 2 — Tell Python where your Census file is

Download and unzip the Census data first. Set DATA_FILE to the CSV containing the actual data, rather than the metadata or notes file.

Change the example path below to your own path.

In [ ]:
# Replace this example with the location of your English Census CSV.
DATA_FILE = Path("data/2021_census_new_brunswick.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"File not found: {DATA_FILE.resolve()}\n"
        "Update DATA_FILE to the location of your unzipped Census CSV."
    )

print("Input file:", DATA_FILE.resolve())

Step 3 — Read only the two geographies we need

We will use:
- Saint John city, rather than the larger metropolitan area.
- New Brunswick, as our comparison.
Saint John’s census subdivision code is 1301006. Census subdivision

The code reads the CSV in smaller pieces so that it can handle a large file without loading everything into memory.

In [ ]:
GEOGRAPHIES = {
    "2021A00051301006": "Saint John city",
    "2021A000213": "New Brunswick",
}

REQUIRED_COLUMNS = {
    "DGUID",
    "CHARACTERISTIC_ID",
    "CHARACTERISTIC_NAME",
    "C1_COUNT_TOTAL",
}

def read_selected_geographies(path):
    # Try the common encodings used in Census downloads.
    for encoding in ["utf-8-sig", "cp1252"]:
        selected_chunks = []

        try:
            reader = pd.read_csv(
                path,
                encoding=encoding,
                dtype=str,
                chunksize=100_000,
                keep_default_na=False,
            )

            for chunk in reader:
                chunk.columns = chunk.columns.str.strip()

                missing = REQUIRED_COLUMNS - set(chunk.columns)
                if missing:
                    raise ValueError(
                        f"Required columns are missing: {sorted(missing)}\n"
                        f"Available columns: {list(chunk.columns)}\n"
                        "Use the English comprehensive Census Profile CSV."
                    )

                chunk["DGUID"] = chunk["DGUID"].str.strip()

                selected = chunk[
                    chunk["DGUID"].isin(GEOGRAPHIES)
                ].copy()

                if not selected.empty:
                    selected_chunks.append(selected)

            if not selected_chunks:
                raise ValueError(
                    "Neither requested geography was found. "
                    "Check that the file contains New Brunswick province "
                    "and Saint John city."
                )

            return pd.concat(selected_chunks, ignore_index=True)

        except UnicodeDecodeError:
            continue

    raise ValueError("The CSV could not be decoded using the expected encodings.")

census = read_selected_geographies(DATA_FILE)

missing_geographies = set(GEOGRAPHIES) - set(census["DGUID"])
if missing_geographies:
    raise ValueError(
        f"The file does not contain these DGUIDs: {missing_geographies}"
    )

print("Rows loaded:", len(census))
display(
    census[["DGUID"]]
    .drop_duplicates()
    .assign(Geography=lambda table: table["DGUID"].map(GEOGRAPHIES))
)

Step 4 — Clean the labels and numerical values

This removes extra spaces from variable names and converts values into numbers.

Suppressed or unavailable values remain missing. We do not replace them with zero. If the file includes statistical symbols, we retain them and avoid using flagged values in the calculations.

In [ ]:
census["characteristic"] = (
    census["CHARACTERISTIC_NAME"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

census["characteristic_id"] = pd.to_numeric(
    census["CHARACTERISTIC_ID"],
    errors="coerce",
)

census["value"] = pd.to_numeric(
    census["C1_COUNT_TOTAL"]
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce",
)

# Preserve the original symbols, where available.
symbol_column = "SYMBOL (C1)"

if symbol_column in census.columns:
    census["value_symbol"] = census[symbol_column].str.strip()

    # Conservatively exclude flagged entries from calculation.
    flagged = census["value_symbol"].ne("")
    census.loc[flagged, "value"] = float("nan")
else:
    census["value_symbol"] = ""

census = census.sort_values(
    ["DGUID", "characteristic_id"]
).reset_index(drop=True)

# Duplicate records could otherwise lead to ambiguous results.
duplicates = census.duplicated(
    ["DGUID", "characteristic_id"],
    keep=False,
)

if duplicates.any():
    display(
        census.loc[
            duplicates,
            ["DGUID", "CHARACTERISTIC_ID", "characteristic", "value"],
        ]
    )
    raise ValueError(
        "Duplicate geography-characteristic records were found. "
        "Check whether multiple copies of the data were combined."
    )

display(census[["DGUID", "characteristic", "value"]].head())

Step 5 — Check the relevant Census variables

Before calculating anything, this cell displays the rows related to housing, income and shelter costs.

This is a useful manual check: we should see the variables we intended to use.

In [ ]:
search_terms = (
    r"structural type|Single-detached|Apartment|"
    r"period of construction|1960 or before|1961 to 1980|"
    r"dwelling condition|Major repairs|"
    r"household by tenure|households by tenure|"
    r"^Owner$|^Renter$|"
    r"Median after-tax income of household|"
    r"shelter-cost-to-income ratio|"
    r"Spending 30% or more"
)

variables_to_check = census[
    census["DGUID"].eq("2021A00051301006")
    & census["characteristic"].str.contains(
        search_terms,
        case=False,
        regex=True,
        na=False,
    )
]

display(
    variables_to_check[
        ["characteristic_id", "characteristic", "value", "value_symbol"]
    ]
)

Step 6 — Define how each indicator is calculated

For percentages, we use the denominator belonging to that Census section.

For example:

Older dwellings (%) =
100 *
((Dwellings built in 1960 or before+Dwellings built in 1961–1980)/
Occupied dwellings classified by construction period)


The functions below also record which Census rows were used. If a label matches multiple rows, the code stops instead of choosing one silently.

In [ ]:
audit_records = []

def get_row(dguid, pattern, data=None):
    """Find exactly one Census characteristic using a full-label pattern."""
    source = census if data is None else data

    matches = source[
        source["DGUID"].eq(dguid)
        & source["characteristic"].str.fullmatch(
            pattern,
            case=False,
            na=False,
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one match for {pattern!r} in {GEOGRAPHIES[dguid]}, "
            f"but found {len(matches)}. Review the labels in Step 5."
        )

    row = matches.iloc[0]

    audit_records.append({
        "Geography": GEOGRAPHIES[dguid],
        "Characteristic ID": row["CHARACTERISTIC_ID"],
        "Census characteristic": row["characteristic"],
        "Value": row["value"],
        "Symbol": row["value_symbol"],
    })

    return row


def get_value(dguid, pattern):
    return get_row(dguid, pattern)["value"]


def percentage(numerator, denominator):
    if pd.isna(numerator) or pd.isna(denominator) or denominator <= 0:
        return float("nan")
    return 100 * numerator / denominator


def get_shelter_burden(dguid):
    """
    Select the affordability count within its specific Census section.
    Similar labels also appear in other sections of the Profile.
    """
    denominator_row = get_row(
        dguid,
        r"Total - Owner and tenant households with household total income "
        r"greater than zero, in non-farm, non-reserve private dwellings "
        r"by shelter-cost-to-income ratio - 25% sample data",
    )

    geography_rows = census[
        census["DGUID"].eq(dguid)
    ].sort_values("characteristic_id")

    following_rows = geography_rows[
        geography_rows["characteristic_id"]
        > denominator_row["characteristic_id"]
    ]

    # End the section at the next Census "Total -" heading.
    next_total = following_rows[
        following_rows["characteristic"].str.startswith("Total -")
    ]

    if not next_total.empty:
        following_rows = following_rows[
            following_rows["characteristic_id"]
            < next_total.iloc[0]["characteristic_id"]
        ]

    numerator_row = get_row(
        dguid,
        r"Spending 30% or more of income on shelter costs",
        data=following_rows,
    )

    return percentage(
        numerator_row["value"],
        denominator_row["value"],
    )

Step 7 — Build the comparison table

The apartment percentage combines the three Census apartment categories: apartments in a duplex, apartments in buildings below five storeys, and apartments in buildings with five or more storeys.

The income row is explicitly labelled 2020, because that is the income reference year.

In [ ]:
def calculate_indicators(dguid):
    dwelling_total = get_value(
        dguid,
        r"Total - Occupied private dwellings by structural type of dwelling "
        r"- 100% data",
    )

    detached = get_value(dguid, r"Single-detached house")

    apartment_counts = [
        get_value(dguid, r"Apartment or flat in a duplex"),
        get_value(dguid, r"Apartment in a building that has fewer than five storeys"),
        get_value(dguid, r"Apartment in a building that has five or more storeys"),
    ]

    # Require all components; do not silently ignore missing categories.
    apartments = pd.Series(apartment_counts).sum(min_count=3)

    construction_total = get_value(
        dguid,
        r"Total - Occupied private dwellings by period of construction "
        r"- 25% sample data",
    )

    older_dwellings = pd.Series([
        get_value(dguid, r"1960 or before"),
        get_value(dguid, r"1961 to 1980"),
    ]).sum(min_count=2)

    condition_total = get_value(
        dguid,
        r"Total - Occupied private dwellings by dwelling condition "
        r"- 25% sample data",
    )

    major_repairs = get_value(dguid, r"Major repairs needed")

    tenure_total = get_value(
        dguid,
        r"Total - Private households by tenure - 25% sample data",
    )

    owners = get_value(dguid, r"Owner")
    renters = get_value(dguid, r"Renter")

    income = get_value(
        dguid,
        r"Median after-tax income of household in 2020 \(\$\)",
    )

    return {
        "Occupied private dwellings (number)": dwelling_total,
        "Single-detached dwellings (%)": percentage(detached, dwelling_total),
        "Apartment dwellings (%)": percentage(apartments, dwelling_total),
        "Dwellings built before 1981 (%)": percentage(
            older_dwellings, construction_total
        ),
        "Dwellings requiring major repairs (%)": percentage(
            major_repairs, condition_total
        ),
        "Owner households (%)": percentage(owners, tenure_total),
        "Renter households (%)": percentage(renters, tenure_total),
        "Median after-tax household income, 2020 (CAD)": income,
        "Households spending at least 30% on shelter (%)": (
            get_shelter_burden(dguid)
        ),
    }


audit_records.clear()

baseline_table = pd.DataFrame({
    name: calculate_indicators(dguid)
    for dguid, name in GEOGRAPHIES.items()
})

baseline_table.index.name = "Indicator"

display(baseline_table.round(1))

Step 8 — Display a cleaner version and check missing results

Counts and income appear without decimal places. Percentages appear with one decimal place.

In [ ]:
formatted_table = baseline_table.copy().astype(object)

for indicator in baseline_table.index:
    decimals = 1 if indicator.endswith("(%)") else 0

    formatted_table.loc[indicator] = baseline_table.loc[indicator].map(
        lambda value: (
            "Unavailable"
            if pd.isna(value)
            else f"{value:,.{decimals}f}"
        )
    )

display(formatted_table)

if baseline_table.isna().any().any():
    print(
        "Some indicators are unavailable. "
        "Review the original values and symbols before interpreting the table."
    )
else:
    print("All proposed indicators were calculated.")

print(
    "\nNotes:\n"
    "- Housing characteristics refer to the 2021 Census.\n"
    "- Household income refers to 2020 and is expressed in nominal CAD.\n"
    "- Percentages use their corresponding Census section denominators.\n"
    "- Census rounding can cause small differences in totals.\n"
    "- Shelter-cost burden measures housing affordability, not energy burden."
)

Step 9 — save the results

Run this cell if you want CSV files. It saves the numerical table and a separate record of the Census rows used, so we can check the calculations later.

In [ ]:
OUTPUT_FOLDER = Path("outputs")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

baseline_table.to_csv(
    OUTPUT_FOLDER / "saint_john_housing_baseline_2021.csv",
    encoding="utf-8-sig",
)

audit_table = pd.DataFrame(audit_records).drop_duplicates()

audit_table.to_csv(
    OUTPUT_FOLDER / "saint_john_housing_baseline_source_rows.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Results saved to:", OUTPUT_FOLDER.resolve())